# Mini-Project 1: UBOS District Population Forecaster

This notebook implements 5-year population projections (2025–2029) across 5 Ugandan districts (Kampala, Wakiso, Gulu, Mbarara, Jinja) to support classroom planning for primary education.

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))
from src.forecaster import (
    DistrictPopulation,
    LinearForecaster,
    CAGRForecaster,
    FibonacciRatioForecaster,
    evaluate_metrics,
    bootstrap_residuals,
    calculate_classroom_needs
)
from src.utils import set_plot_style

set_plot_style()

In [ ]:
years = np.arange(2015, 2025)
districts_data = {
    "Kampala": [1200, 1250, 1300, 1350, 1420, 1500, 1580, 1650, 1720, 1800],
    "Wakiso":  [ 950, 1000, 1070, 1150, 1220, 1300, 1390, 1480, 1570, 1670],
    "Gulu":    [ 320,  330,  345,  360,  375,  390,  410,  430,  455,  480],
    "Mbarara": [ 470,  485,  505,  525,  550,  575,  605,  635,  670,  705],
    "Jinja":   [ 450,  462,  475,  490,  505,  520,  538,  555,  575,  595]
}

districts = {name: DistrictPopulation(name, years, pop) for name, pop in districts_data.items()}
for d in districts.values():
    print(repr(d))

## Task 2: Statistical Comparison (`statistics` vs. `NumPy`)

- **`statistics.variance`**: Calculates sample variance dividing by $N - 1$ (`ddof=1`).
- **`np.var`**: Calculates population variance dividing by $N$ (`ddof=0`).
- **Delta-Degrees of Freedom (`ddof`)**: Controls the variance denominator ($N - \text{ddof}$). Setting `ddof=1` in NumPy aligns its result with `statistics.variance`.

In [ ]:
cagr_results = {}
for name, dp in districts.items():
    cagr_results[name] = dp.compute_cagr() * 100.0
    print(f"{name:10s} | CAGR: {cagr_results[name]:.2f}%")

fastest = max(cagr_results, key=cagr_results.get)
print(f"\nFastest Growing District: {fastest} ({cagr_results[fastest]:.2f}% annual growth)")

In [ ]:
train_mask = years <= 2021
test_mask = years >= 2022
years_train, years_test = years[train_mask], years[test_mask]

best_models = {}

for name, dp in districts.items():
    pop_train, pop_test = dp.population[train_mask], dp.population[test_mask]
    
    models = {
        "Linear": LinearForecaster(),
        "CAGR": CAGRForecaster(),
        "Fibonacci": FibonacciRatioForecaster()
    }
    
    metrics_list = []
    for m_name, model in models.items():
        model.fit(years_train, pop_train)
        preds = model.predict(years_test)
        m = evaluate_metrics(pop_test, preds)
        m["Model"] = m_name
        metrics_list.append(m)
        
    df_metrics = pd.DataFrame(metrics_list).set_index("Model")
    print(f"=== {name} Validation Metrics ===")
    print(df_metrics.round(2))
    
    best_m_name = df_metrics["RMSE"].idxmin()
    best_models[name] = best_m_name
    print(f"--> Best Model: {best_m_name}\n")

In [ ]:
years_forecast = np.arange(2025, 2030)
planning_summary = []

for name, dp in districts.items():
    b_name = best_models[name]
    model_cls = {"Linear": LinearForecaster, "CAGR": CAGRForecaster, "Fibonacci": FibonacciRatioForecaster}[b_name]
    
    full_model = model_cls().fit(dp.years, dp.population)
    forecast_vals = full_model.predict(years_forecast)
    
    needs_2024 = calculate_classroom_needs(dp.population[-1])
    needs_2029 = calculate_classroom_needs(forecast_vals[-1])
    
    planning_summary.append({
        "District": name,
        "Best Model": b_name,
        "2024 Pop (k)": dp.population[-1],
        "2029 Pop (k)": round(forecast_vals[-1], 1),
        "2029 Pupils": int(needs_2029["pupils"]),
        "Classrooms Needed (2029)": needs_2029["classrooms"] - needs_2024["classrooms"]
    })

df_planning = pd.DataFrame(planning_summary)
df_planning

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 11), dpi=120)
axes = axes.flatten()

for idx, (name, dp) in enumerate(districts.items()):
    ax = axes[idx]
    b_name = best_models[name]
    model_cls = {"Linear": LinearForecaster, "CAGR": CAGRForecaster, "Fibonacci": FibonacciRatioForecaster}[b_name]
    
    model = model_cls()
    fc, lb, ub = bootstrap_residuals(model, dp.years, dp.population, years_forecast, n_resamples=1000, seed=42)
    fitted_hist = model.fit(dp.years, dp.population).predict(dp.years)
    
    ax.plot(dp.years, dp.population, 'o-', label='Actual Data', color='#2c3e50', lw=2)
    ax.plot(dp.years, fitted_hist, '--', label=f'Fitted ({b_name})', color='#2980b9')
    ax.plot(years_forecast, fc, 's-', label='Forecast (2025-2029)', color='#e74c3c', lw=2)
    ax.fill_between(years_forecast, lb, ub, color='#e74c3c', alpha=0.2, label='95% Bootstrap Interval')
    ax.axvline(2021.5, color='gray', linestyle=':', label='Train/Test Split')
    
    ax.set_title(f"{name} District ({b_name} Model)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Population (k)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

## Critical Assessment: Fibonacci-Ratio Model

The Fibonacci-ratio forecaster scales population using $F(k+1) / F(k)$. Since $F(k+1)/F(k) \to \phi \approx 1.618$ (the Golden Ratio), this model forces an explosive **61.8% annual population growth rate**. 

* **Demographic Defense:** Unrealistic for human populations (which typically grow at 1%–4% annually).
* **Validation Outcome:** Produces severe prediction errors and should be rejected for policy and infrastructure planning.